# Employee Attrition Analysis: From Symptoms to Strategy
### A People Analytics Case Study Using IBM HR Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabriel-ortizf/attrition-ibm/blob/main/IBM_Attrition_Analysis_v2.ipynb)

**Author:** Gabriel Ortiz Francisco  
**Stack:** Python · pandas · statsmodels · seaborn  
**Dataset:** IBM HR Analytics Employee Attrition & Performance (Kaggle)  
**N:** 1,470 employees

---

## The Business Problem

The organization is losing **16.1% of its workforce annually** — nearly 1 in 6 employees.  
At an industry-standard replacement cost of 1.5× annual salary, this translates to an estimated **$27.7M in avoidable costs**.

This analysis answers three questions:
1. **Who** is leaving? (demographic and role profile)
2. **Why** are they leaving? (validated driver analysis)
3. **Who is next?** (risk scoring and intervention targeting)

The goal is not to describe attrition — it is to give decision-makers a framework to prevent it.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from scipy.stats import chi2
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Visual style
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid', font_scale=1.1)

# Color palette
COLOR_STAY   = '#4A90D9'
COLOR_LEAVE  = '#E05C5C'
COLOR_ACCENT = '#F5A623'
COLOR_DARK   = '#2C3E50'

# Load data directly from GitHub
url = 'https://raw.githubusercontent.com/gabriel-ortizf/attrition-ibm/main/data/WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = pd.read_csv(url)

# Encode target
df['att_bin'] = (df['Attrition'] == 'Yes').astype(int)
df['OverTime_bin'] = (df['OverTime'] == 'Yes').astype(int)

# Drop constant columns
df.drop(columns=['EmployeeCount', 'Over18', 'StandardHours'], inplace=True)

print(f'Dataset: {df.shape[0]:,} employees · {df.shape[1]} variables')
print(f'Attrition rate: {df["att_bin"].mean()*100:.1f}%')
print(f'Employees who left: {df["att_bin"].sum():,}')

---
## 1. Who is leaving?

Before modeling drivers, we need to understand the profile of departing employees across roles, departments, and demographics.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Attrition Rate by Role, Department and Marital Status', 
             fontsize=14, fontweight='bold', y=1.02)

# By Job Role
role_att = df.groupby('JobRole')['att_bin'].mean().sort_values() * 100
colors = [COLOR_LEAVE if x > 20 else COLOR_STAY for x in role_att]
role_att.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title('By Job Role', fontweight='bold')
axes[0].set_xlabel('Attrition Rate (%)')
axes[0].axvline(df['att_bin'].mean()*100, color=COLOR_DARK, 
                linestyle='--', linewidth=1, label=f'Overall avg ({df["att_bin"].mean()*100:.1f}%)')
axes[0].legend(fontsize=8)
for i, v in enumerate(role_att):
    axes[0].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=8)

# By Department
dept_att = df.groupby('Department')['att_bin'].mean().sort_values() * 100
dept_att.plot(kind='barh', ax=axes[1], 
              color=[COLOR_LEAVE if x > 16 else COLOR_STAY for x in dept_att])
axes[1].set_title('By Department', fontweight='bold')
axes[1].set_xlabel('Attrition Rate (%)')
axes[1].axvline(df['att_bin'].mean()*100, color=COLOR_DARK, linestyle='--', linewidth=1)
for i, v in enumerate(dept_att):
    axes[1].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)

# By Marital Status
mar_att = df.groupby('MaritalStatus')['att_bin'].mean().sort_values() * 100
mar_att.plot(kind='barh', ax=axes[2],
             color=[COLOR_LEAVE if x > 16 else COLOR_STAY for x in mar_att])
axes[2].set_title('By Marital Status', fontweight='bold')
axes[2].set_xlabel('Attrition Rate (%)')
axes[2].axvline(df['att_bin'].mean()*100, color=COLOR_DARK, linestyle='--', linewidth=1)
for i, v in enumerate(mar_att):
    axes[2].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig1_who_is_leaving.png', bbox_inches='tight')
plt.show()

print('\nKey observation: Sales Representatives (39.8%) leave at 2.5x the company average.')
print('Single employees leave at twice the rate of married/divorced peers.')

In [ ]:
# Numeric profile comparison: who stays vs who leaves
profile_vars = ['Age', 'MonthlyIncome', 'YearsAtCompany', 
                'YearsInCurrentRole', 'TotalWorkingYears', 'DistanceFromHome']

profile = df.groupby('Attrition')[profile_vars].mean().T
profile['Difference'] = profile['Yes'] - profile['No']
profile['Diff_%'] = (profile['Difference'] / profile['No'] * 100).round(1)

print('=== PROFILE: WHO STAYS vs WHO LEAVES ===')
print(f'{"Variable":25} {"Stayed":>10} {"Left":>10} {"Δ":>10} {"Δ%":>8}')
print('-' * 65)
for var in profile_vars:
    stayed = profile.loc[var, 'No']
    left = profile.loc[var, 'Yes']
    diff = profile.loc[var, 'Difference']
    diffp = profile.loc[var, 'Diff_%']
    print(f'{var:25} {stayed:>10.1f} {left:>10.1f} {diff:>+10.1f} {diffp:>7.1f}%')

print()
print('Employees who leave are on average 4 years younger, earn $2,046/month less,')
print('and have spent 2.3 fewer years at the company.')

---
## 2. Why are they leaving?

Descriptive rates tell us *where* attrition is happening. To understand *why*, we need to quantify the independent contribution of each factor while controlling for all others.

We use **logistic regression with standardized coefficients** — this allows direct comparison of effect sizes across variables measured on different scales.

The **Odds Ratio (OR)** tells us how much each factor multiplies the probability of leaving:
- OR = 2.0 means the employee is 2× more likely to leave
- OR < 1.0 means the factor is protective (reduces attrition risk)

In [ ]:
# Logistic regression — driver analysis
model_features = {
    'OverTime_bin':             'Overtime',
    'Single':                   'Single (marital)',
    'NumCompaniesWorked':       'Companies worked',
    'DistanceFromHome':         'Distance from home',
    'WorkLifeBalance':          'Work-life balance',
    'JobInvolvement':           'Job involvement',
    'JobSatisfaction':          'Job satisfaction',
    'EnvironmentSatisfaction':  'Environment satisfaction',
    'Age':                      'Age',
    'YearsInCurrentRole':       'Years in current role',
    'MonthlyIncome':            'Monthly income',
}

df['Single'] = (df['MaritalStatus'] == 'Single').astype(int)

X = df[list(model_features.keys())]
y = df['att_bin']

# Standardize features for comparable coefficients
scaler_sm = StandardScaler()
X_std = pd.DataFrame(scaler_sm.fit_transform(X), columns=list(model_features.keys()))
X_const = sm.add_constant(X_std)

# Fit logistic regression with statsmodels
logit_model = sm.Logit(y.values, X_const)
result = logit_model.fit(maxiter=200, disp=False)

# Extract results
coef_df = pd.DataFrame({
    'Feature': ['Intercept'] + list(model_features.values()),
    'Coefficient': result.params.values,
    'Odds_Ratio': np.exp(result.params.values),
    'p_value': result.pvalues.values,
    'CI_low': np.exp(result.conf_int().iloc[:,0].values),
    'CI_high': np.exp(result.conf_int().iloc[:,1].values),
}).iloc[1:].sort_values('Coefficient', ascending=False).reset_index(drop=True)

# Model fit
print('=== LOGISTIC REGRESSION RESULTS (statsmodels) ===')
print(f'Log-likelihood: {result.llf:.1f}')
print(f'Pseudo R² (McFadden): {result.prsquared:.3f}')
print(f'AIC: {result.aic:.1f}')
print()
print(f'{"Factor":28} {"Std.Coef":>9} {"OR":>7} {"95% CI":>18} {"p":>8} {"Sig":>5}')
print('-' * 78)
for _, row in coef_df.iterrows():
    sig = '***' if row['p_value'] < .001 else '**' if row['p_value'] < .01 else '*' if row['p_value'] < .05 else ''
    print(f'{row["Feature"]:28} {row["Coefficient"]:>9.3f} {row["Odds_Ratio"]:>7.3f} '
          f'[{row["CI_low"]:5.2f}, {row["CI_high"]:5.2f}] {row["p_value"]:>8.3f} {sig:>5}')

In [ ]:
# Visualize drivers
fig, ax = plt.subplots(figsize=(10, 6))

coef_sorted = coef_df.sort_values('Coefficient')
colors = [COLOR_LEAVE if c > 0 else COLOR_STAY for c in coef_sorted['Coefficient']]

bars = ax.barh(coef_sorted['Feature'], coef_sorted['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color=COLOR_DARK, linewidth=1.2)
ax.set_title('What Drives Attrition?\nStandardized Logistic Regression Coefficients', 
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Standardized Coefficient\n(positive = increases attrition risk, negative = protective)')

# Value labels
for bar, val in zip(bars, coef_sorted['Coefficient']):
    offset = 0.02 if val >= 0 else -0.02
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height()/2, 
            f'{val:+.2f}', va='center', ha=ha, fontsize=9)

leave_patch = mpatches.Patch(color=COLOR_LEAVE, label='Increases attrition risk')
stay_patch = mpatches.Patch(color=COLOR_STAY, label='Reduces attrition risk')
ax.legend(handles=[leave_patch, stay_patch], loc='lower right')

plt.tight_layout()
plt.savefig('fig2_drivers.png', bbox_inches='tight')
plt.show()

### 📋 Driver Analysis — Executive Summary

| Driver | Odds Ratio | Interpretation |
|---|---|---|
| **Overtime** | 2.10 | Overtime workers are **2.1× more likely** to leave |
| **Single (marital)** | 1.57 | Single employees are **1.6× more likely** to leave |
| **Companies worked** | 1.38 | Each additional employer raises risk by **38%** |
| **Monthly income** | 0.62 | Higher income is **strongly protective** |
| **Years in current role** | 0.62 | Role stability reduces risk significantly |

**Key insight:** Overtime is the dominant risk factor — but it doesn't operate in isolation. Its effect is amplified by low income and personal vulnerability (being single). This suggests the problem is structural, not individual.

---
## 3. The Compounding Effect: When Risk Factors Stack

Individual risk factors are informative. But attrition rarely has a single cause — it results from the **accumulation** of stressors. Here we examine how overtime interacts with the two highest-risk profiles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('The Compounding Effect: When Risk Factors Stack', 
             fontsize=13, fontweight='bold')

# Heatmap 1: Overtime × Marital Status
ct1 = df.groupby(['MaritalStatus', 'OverTime'])['att_bin'].mean().unstack() * 100
ct1.columns = ['Standard hours', 'Overtime']
sns.heatmap(ct1, annot=True, fmt='.1f', cmap='RdYlGn_r', 
            ax=axes[0], cbar_kws={'label': 'Attrition Rate (%)'},
            vmin=0, vmax=55)
axes[0].set_title('Overtime × Marital Status', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Marital Status')

# Heatmap 2: Age group × Income level
df['age_grp'] = pd.cut(df['Age'], bins=[17,30,40,60], labels=['Under 30','30–40','Over 40'])
df['inc_grp'] = pd.qcut(df['MonthlyIncome'], q=3, labels=['Low income','Mid income','High income'])
ct2 = df.groupby(['age_grp','inc_grp'])['att_bin'].mean().unstack() * 100
sns.heatmap(ct2, annot=True, fmt='.1f', cmap='RdYlGn_r',
            ax=axes[1], cbar_kws={'label': 'Attrition Rate (%)'},
            vmin=0, vmax=55)
axes[1].set_title('Age Group × Income Level', fontweight='bold')
axes[1].set_xlabel('Income Level')
axes[1].set_ylabel('Age Group')

plt.tight_layout()
plt.savefig('fig3_heatmaps.png', bbox_inches='tight')
plt.show()

single_ot = df[(df['MaritalStatus']=='Single')&(df['OverTime']=='Yes')]['att_bin'].mean()*100
young_low = df[(df['age_grp']=='Under 30')&(df['inc_grp']=='Low income')]['att_bin'].mean()*100
print(f'Single + Overtime: {single_ot:.1f}% attrition — nearly 1 in 2 employees')
print(f'Under 30 + Low income: {young_low:.1f}% attrition — highest risk demographic')

---
## 4. The Friction Index: A Composite Risk Score

Individual predictors are useful but fragmented. A composite score enables HR teams to **rank employees by attrition risk** and intervene proactively.

The Friction Index is constructed using the **standardized coefficients from the logistic regression** — not arbitrary weights. Each factor's contribution is proportional to its validated predictive power.

> **Friction Index = 0.74×Overtime + 0.45×Single + 0.32×NumCompanies + 0.27×Distance − 0.48×Income − 0.48×YearsInRole − 0.39×Age**

*Variables are normalized before combining.*

In [ ]:
# Build validated Friction Index
df['inc_norm']  = df['MonthlyIncome'] / df['MonthlyIncome'].max()
df['role_norm'] = df['YearsInCurrentRole'] / df['YearsInCurrentRole'].max()
df['age_norm']  = df['Age'] / df['Age'].max()
df['dist_norm'] = df['DistanceFromHome'] / df['DistanceFromHome'].max()
df['comp_norm'] = df['NumCompaniesWorked'] / df['NumCompaniesWorked'].max()

df['Friction_Index'] = (
     0.74 * df['OverTime_bin']
    + 0.45 * df['Single']
    + 0.32 * df['comp_norm']
    + 0.27 * df['dist_norm']
    - 0.48 * df['inc_norm']
    - 0.48 * df['role_norm']
    - 0.39 * df['age_norm']
)

# Validate
threshold = df['Friction_Index'].quantile(0.75)
df['risk_zone'] = df['Friction_Index'] >= threshold

red_att  = df[df['risk_zone']]['att_bin'].mean() * 100
safe_att = df[~df['risk_zone']]['att_bin'].mean() * 100
at_risk_active = df[df['risk_zone'] & (df['att_bin']==0)]

print('=== FRICTION INDEX VALIDATION ===')
print(f'Red zone (top 25%) attrition rate:  {red_att:.1f}%')
print(f'Safe zone (bottom 75%) attrition:   {safe_att:.1f}%')
print(f'Risk multiplier:                     {red_att/safe_att:.1f}×')
print()
print(f'Currently employed employees in red zone: {len(at_risk_active)}')
avg_salary = df['MonthlyIncome'].mean() * 12
print(f'Estimated cost if they leave: ${len(at_risk_active) * avg_salary * 1.5:,.0f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('The Friction Index: Identifying At-Risk Employees', 
             fontsize=13, fontweight='bold')

# Distribution plot
sns.kdeplot(df[df['att_bin']==0]['Friction_Index'], ax=axes[0],
            fill=True, color=COLOR_STAY, alpha=0.5, label='Stayed')
sns.kdeplot(df[df['att_bin']==1]['Friction_Index'], ax=axes[0],
            fill=True, color=COLOR_LEAVE, alpha=0.5, label='Left')
axes[0].axvline(threshold, color=COLOR_ACCENT, linestyle='--', linewidth=2,
                label=f'Risk threshold (top 25%)')
axes[0].set_title('Risk Distribution by Attrition Status', fontweight='bold')
axes[0].set_xlabel('Friction Index')
axes[0].set_ylabel('Employee Density')
axes[0].legend()
axes[0].annotate('Red Zone', xy=(threshold+0.02, axes[0].get_ylim()[1]*0.9),
                 color=COLOR_ACCENT, fontweight='bold', fontsize=10)

# ROC curve
# ROC curve — computed manually without sklearn
def compute_roc(y_true, y_score):
    thresholds = np.sort(np.unique(y_score))[::-1]
    tpr_list, fpr_list = [0], [0]
    P = y_true.sum()
    N = len(y_true) - P
    for t in thresholds:
        pred = (y_score >= t).astype(int)
        tp = ((pred==1) & (y_true==1)).sum()
        fp = ((pred==1) & (y_true==0)).sum()
        tpr_list.append(tp/P)
        fpr_list.append(fp/N)
    tpr_list.append(1); fpr_list.append(1)
    return np.array(fpr_list), np.array(tpr_list)

def compute_auc(fpr, tpr):
    return np.trapezoid(tpr, fpr) if hasattr(np, 'trapezoid') else np.trapz(tpr, fpr)

fpr, tpr = compute_roc(df['att_bin'].values, df['Friction_Index'].values)
auc_score = compute_auc(fpr, tpr)
axes[1].plot(fpr, tpr, color=COLOR_LEAVE, linewidth=2,
             label=f'Friction Index (AUC = {auc_score:.2f})')
axes[1].plot([0,1],[0,1], 'k--', linewidth=1, label='Random (AUC = 0.50)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color=COLOR_LEAVE)
axes[1].set_title('ROC Curve: Predictive Power', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig4_friction_index.png', bbox_inches='tight')
plt.show()

print(f'Friction Index AUC: {auc_score:.3f}')
print('AUC > 0.70 indicates good discriminative ability.')
print('The index correctly separates stayers from leavers in 7 out of 10 cases.')

---
## 5. The At-Risk Profile

Combining the Friction Index with role and department data reveals **where the next wave of attrition is most likely to come from**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Where Is the Next Wave of Attrition Coming From?', 
             fontsize=13, fontweight='bold')

# Average friction index by job role
role_friction = df[df['att_bin']==0].groupby('JobRole')['Friction_Index'].mean().sort_values(ascending=True)
colors = [COLOR_LEAVE if v > threshold else COLOR_STAY for v in role_friction]
role_friction.plot(kind='barh', ax=axes[0], color=colors)
axes[0].axvline(threshold, color=COLOR_ACCENT, linestyle='--', linewidth=1.5,
                label='Risk threshold')
axes[0].set_title('Mean Friction Index by Role\n(Current employees only)', fontweight='bold')
axes[0].set_xlabel('Mean Friction Index')
axes[0].legend(fontsize=8)

# Count of red-zone employees by department
dept_risk = df[(df['att_bin']==0) & df['risk_zone']].groupby('Department').size()
dept_total = df[df['att_bin']==0].groupby('Department').size()
dept_pct = (dept_risk / dept_total * 100).sort_values(ascending=True)
dept_pct.plot(kind='barh', ax=axes[1], 
              color=[COLOR_LEAVE if v > 25 else COLOR_STAY for v in dept_pct])
axes[1].set_title('% of Current Employees in Red Zone\nby Department', fontweight='bold')
axes[1].set_xlabel('% in Risk Zone')
for i, v in enumerate(dept_pct):
    axes[1].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig5_at_risk_profile.png', bbox_inches='tight')
plt.show()

---
## 6. Executive Summary & Strategic Recommendations

### The Business Case in Numbers

| Metric | Value |
|---|---|
| Annual attrition rate | 16.1% |
| Employees who left | 237 |
| Estimated cost of past attrition | $27.7M |
| Employees currently in red zone | 226 |
| Estimated cost if red zone leaves | $26.5M |
| **Total at-risk exposure** | **$54.2M** |

### The Three Root Causes

**1. Structural overwork (OR = 2.10)**  
Overtime is the single strongest predictor of attrition. It is not a compensation problem — it is a staffing problem. Employees are working overtime because roles are understaffed, not because they want to.

**2. Life-stage vulnerability (OR = 1.57)**  
Single employees — who lack external support structures — are 1.6× more likely to leave. When combined with overtime, their attrition rate reaches 49.6%. The organization is disproportionately burning out its youngest, most mobile talent.

**3. Compensation below market (OR = 0.62 protective)**  
Employees who leave earn $2,046/month less than those who stay. The income gap is widest in Sales Representative roles — the highest attrition position in the company.

### Three Prioritized Recommendations

**Priority 1 — Staffing audit in Sales (highest ROI)**  
Sales Representatives have a 39.8% attrition rate and high overtime dependency. A headcount analysis to reduce structural overtime in this role would have the highest impact per dollar invested.

**Priority 2 — Compensation review for junior roles**  
Employees under 30 with low income have a 35% attrition rate. A targeted pay-parity review for entry-to-mid-level positions — not a broad raise — would reduce the income component of the Friction Index.

**Priority 3 — Proactive retention for red-zone employees**  
219 current employees score above the risk threshold. HR should prioritize stay interviews and career development conversations with this group before they reach the resignation decision.

In [ ]:
# Final summary statistics
avg_salary = df['MonthlyIncome'].mean() * 12
n_left = df['att_bin'].sum()
n_red_zone = len(df[(df['att_bin']==0) & df['risk_zone']])

print('=== FINAL BUSINESS CASE SUMMARY ===')
print(f'Annual attrition rate:              {df["att_bin"].mean()*100:.1f}%')
print(f'Employees who left:                 {n_left:,}')
print(f'Average annual salary:              ${avg_salary:,.0f}')
print(f'Cost of past attrition (1.5x):      ${n_left * avg_salary * 1.5:,.0f}')
print(f'Employees in red zone (active):     {n_red_zone:,}')
print(f'Potential cost if they leave:       ${n_red_zone * avg_salary * 1.5:,.0f}')
print()
print('=== TOP 3 RISK PROFILES ===')
print(f'Single + Overtime:                  {df[(df["Single"]==1)&(df["OverTime"]=="Yes")]["att_bin"].mean()*100:.1f}%')
print(f'Under 30 + Low income:              {df[(df["age_grp"]=="Under 30")&(df["inc_grp"]=="Low income")]["att_bin"].mean()*100:.1f}%')
print(f'Sales Representative:               39.8%')
print()
fpr_f, tpr_f = compute_roc(df['att_bin'].values, df['Friction_Index'].values)
auc_final = compute_auc(fpr_f, tpr_f)
print('Friction Index AUC:', round(auc_final, 3))